# 🧬 BioScreen-QSAR — Demo Pipeline

This notebook walks through the complete BioScreen-QSAR workflow step by step:
1. Data curation
2. Duplicate resolution
3. ECFP descriptor generation
4. Model training & validation
5. Virtual screening

> Make sure you have installed all dependencies: `pip install -r requirements.txt`

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('✅ Imports OK')

## Step 1 — Load and Inspect Raw Data

In [ ]:
from scripts.utils import load_csv

df_raw = load_csv('../data/example_input.csv', smiles_col='SMILES', activity_col='activity')
print(f'Loaded: {len(df_raw)} compounds')
df_raw.head()

## Step 2 — Molecular Standardisation

In [ ]:
from scripts.data_curation import MolecularStandardiser

standardiser = MolecularStandardiser(
    remove_mixtures=True,
    neutralise_charges=True,
    standardise_tautomers=True
)

df_std = standardiser.standardise_dataframe(df_raw, smiles_col='SMILES', activity_col='activity')
print(f'After standardisation: {len(df_std)} compounds')
print(f'Removed: {len(df_raw) - len(df_std)}')
df_std[['SMILES', 'curated_SMILES', 'activity']].head()

## Step 3 — Duplicate Resolution

In [ ]:
from scripts.duplicate_handling import handle_duplicates

df_curated, stats = handle_duplicates(
    df_std,
    smiles_col='curated_SMILES',
    activity_col='activity',
    data_type='binary'
)

print('Duplicate Analysis Stats:')
for k, v in stats.items():
    print(f'  {k:40s}: {v}')

## Step 4 — ECFP Fingerprint Generation

In [ ]:
from scripts.descriptors_ecfp import ECFPGenerator

generator = ECFPGenerator(radius=2, n_bits=1024)  # ECFP4/1024 for demo speed

X, y, valid_smiles = generator.dataframe_to_fp_matrix(
    df_curated,
    smiles_col='curated_SMILES',
    activity_col='activity'
)

print(f'Fingerprint matrix shape: {X.shape}')
print(f'Activity distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

# Visualise bit density
plt.figure(figsize=(10, 3))
plt.bar(range(1024), X.mean(axis=0), color='steelblue', width=1)
plt.xlabel('ECFP4 Bit Index')
plt.ylabel('Frequency')
plt.title('ECFP4 Bit Frequency Distribution')
plt.tight_layout()
plt.show()

## Step 5 — Feature Preprocessing & Data Split

In [ ]:
from scripts.preprocessing import FeaturePreprocessor

preprocessor = FeaturePreprocessor(
    variance_threshold=0.0,
    test_size=0.20,
    n_splits=5,
    random_state=42,
    stratify=True
)

X_train, X_test, y_train, y_test = preprocessor.fit_transform(X, y.astype(int))
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## Step 6 — Train a Quick Classification Model (no HPO for demo)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from scripts.validation_metrics import (
    classification_metrics, plot_roc_curve, plot_confusion_matrix
)

model = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = classification_metrics(y_test, y_pred, y_prob)
print('\nExternal test set metrics:')
for k, v in metrics.items():
    print(f'  {k:15s}: {v}')

In [ ]:
plot_roc_curve(y_test, y_prob, model_name='RandomForest (demo)')

In [ ]:
plot_confusion_matrix(y_test, y_pred, model_name='RandomForest (demo)')

## Step 7 — Save and Load Model

In [ ]:
from scripts.model_serialization import save_model, load_model

save_model(model, '../models/demo_rf_classifier.pkl', metadata={
    'algorithm': 'RandomForest',
    'task': 'classification',
    'ecfp_radius': 2,
    'ecfp_n_bits': 1024,
    'auc': metrics['auc']
})

loaded_model = load_model('../models/demo_rf_classifier.pkl')
print(f'Model loaded: {type(loaded_model).__name__}')

## Step 8 — Virtual Screening on Demo Library

In [ ]:
from scripts.virtual_screening import run_virtual_screening

results = run_virtual_screening(
    library_path='../data/library_demo.csv',
    model_path='../models/demo_rf_classifier.pkl',
    preprocessor=preprocessor,
    smiles_col='SMILES',
    task='classification',
    ecfp_radius=2,
    ecfp_n_bits=1024,
    output_path='../results/demo_screening_results.csv'
)

print(f'\nScreened: {len(results)} compounds')
n_active = (results['predicted_class'] == 1).sum()
print(f'Predicted active (P ≥ 0.5): {n_active}')
results.head(10)

In [ ]:
# Probability distribution
plt.figure(figsize=(8, 4))
plt.hist(results['probability_active'], bins=20, color='steelblue', edgecolor='white')
plt.axvline(0.5, color='red', linestyle='--', label='Threshold 0.5')
plt.xlabel('P(active)')
plt.ylabel('Count')
plt.title('Virtual Screening — Distribution of Predicted Probabilities')
plt.legend()
plt.tight_layout()
plt.show()

---
## 🏁 Pipeline Complete!

You have successfully:
- ✅ Curated a molecular dataset
- ✅ Resolved duplicates
- ✅ Generated ECFP fingerprints
- ✅ Trained and validated a classifier
- ✅ Performed virtual screening

For the full pipeline with Bayesian HPO, run:
```bash
streamlit run scripts/app.py
```